# Sprint 1 Live — Service workshops

Use this notebook for four workshops. Each code cell sends requests through the supplied FieldCare FastAPI application. Supported requests use the Module A model path: Chroma retrieval, LangGraph orchestration, and a real OpenRouter generation call. Set `OPENROUTER_API_KEY` in Colab Secrets (or use the hidden prompt) before the first request. Model output varies; validation and clarification branches are deterministic service behavior.

Keep this Live workspace separate from your Campus notebook. You may import a Campus checkpoint in setup, or begin with the supplied service.

**Colab runtime:** These Sprint 1 exercises use FastAPI `TestClient` in-process, so they do not start Uvicorn in the notebook’s foreground cell. Do not add `uvicorn.run()` or an ngrok tunnel; `uvicorn.run()` keeps its cell occupied until the server stops. Later sprints use the Module B `ServiceProcess` helper to start one loopback-only Uvicorn child, check readiness with a timeout, and stop only that owned process. The request cells wait for their real HTTP/provider response; the server itself does not hold the cell open.


## Instructor guides and live decks in Notion

- [LS 1 — Application vs service and REST orientation](https://app.notion.com/p/3ea5a55972ad8125af2edd3fa083a7d9?pvs=204)
- [LS 2 — Extending the scaffold](https://app.notion.com/p/3ea5a55972ad817d8151ff4a5cb6bdcc?pvs=204)
- [LS 3 — Input/output contracts and versioning](https://app.notion.com/p/3ea5a55972ad8159b1ffd860117002d4?pvs=204)
- [LS 4 — Sprint 1 checkpoint](https://app.notion.com/p/3ea5a55972ad81da914cc512dec06b48?pvs=204)


In [ ]:
# Setup: run once in a fresh Colab runtime.
from pathlib import Path
from shutil import copytree
from zipfile import ZipFile
import os
import subprocess
import sys

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path("/content/ai-app-dev_Mod-B")
PROJECT = Path("/content/fieldcare-live")
CAMPUS_CHECKPOINT = ""  # Optional uploaded Campus ZIP

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
else:
    # Colab runtimes persist /content; refresh an existing clone before reinstalling it.
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))  # Make the editable package visible to this running kernel.

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)

if CAMPUS_CHECKPOINT:
    PROJECT.mkdir(parents=True, exist_ok=True)
    with ZipFile(CAMPUS_CHECKPOINT) as archive:
        archive.extractall(PROJECT)
elif not PROJECT.exists():
    copytree(REPO / "examples" / "fieldcare", PROJECT)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()  # Start the app lifespan and initialize its real Chroma/OpenRouter resources.
print("Separate Live workspace:", PROJECT)

## LS01 — Caller and service

Predict the method, path, and status for each call. Then compare your prediction with the actual response. Health reports service readiness; it does not mean a provider is available.

In [ ]:
supported = {
    "question": "Which airflow and filter checks are documented?",
    "equipment_id": "EQ-FC-1002",
}
incomplete = {"question": "Which airflow checks are documented?"}

for label, result in [
    ("health", client.get("/health")),
    ("supported", client.post("/v1/diagnose", json=supported)),
    ("invalid input", client.post("/v1/diagnose", json={})),
    ("missing optional context", client.post("/v1/diagnose", json=incomplete)),
]:
    print(label, "→", result.status_code, result.json())

Trace a supported request in the Files panel: app/routes.py, app/schemas.py, app/service.py, then app/model.py. Note which facts came from the response and which required reading the implementation. Explain why 422 and a successful clarification are different caller outcomes.

## LS02 — Test the extension as a caller

Exchange route contracts with a partner. Send requests that test accepted input, invalid input, missing optional context, and the original technician route. Use the path your partner gives you.

In [ ]:
ROUTE_TO_REVIEW = "/v1/diagnose"  # Replace with your partner's route.
review_requests = [
    ("supported", supported),
    ("invalid", {}),
    ("missing optional context", incomplete),
]
for label, payload in review_requests:
    result = client.post(ROUTE_TO_REVIEW, json=payload)
    print(label, "→", result.status_code, result.json())

Compare each response with the contract card. Generated wording varies, so a single response cannot establish that a prompt improved answer quality. Record one expectation the caller can rely on from the contract and one claim that requires evidence beyond schema validation.

## LS03 — Compatibility and migration

A stricter contract can break an existing caller. Predict whether a 900-character question is accepted by each version, then run the calls. Use the versioned paths you implemented in Campus or set OLD_PATH and NEW_PATH to the routes prepared for your session.


In [ ]:
# Enter the two versioned paths prepared for this workshop.
OLD_PATH = "/v1/dispatch-handover"
NEW_PATH = "/v2/dispatch-handover"
short_request = supported
long_request = {
    "question": "filter " + "x" * 900,
    "equipment_id": "EQ-FC-1002",
}
for path in [OLD_PATH, NEW_PATH]:
    for label, payload in [("short", short_request), ("long", long_request)]:
        result = client.post(path, json=payload)
        print(path, label, "→", result.status_code)


Explain which caller would break if the old agreement were replaced. Inspect the route code and locate each version’s request schema, prompt, and model setting. A new path alone is not proof that those settings are independent.

## LS04 — Demonstrate and review

Choose one public operation. Demonstrate an accepted request and an invalid request. Ask a reviewer where the request is rejected or handled and what observation shows an earlier operation remains available.

In [ ]:
REVIEW_PATH = "/v1/diagnose"
result = client.post(REVIEW_PATH, json=supported)
print("Request:", supported)
print("Response:", result.status_code, result.json())

In your saved notebook notes, record the route and contract you showed, the evidence you observed, one limitation of that evidence, and a repair you made or why no change was needed. Save the notebook and keep the Live ZIP separate from the Campus checkpoint.

## Save your Live checkpoint

In [ ]:
from zipfile import ZIP_DEFLATED

CHECKPOINT_OUT = Path("/content/sprint-1-live-checkpoint.zip")
with ZipFile(CHECKPOINT_OUT, "w", ZIP_DEFLATED) as archive:
    for folder in ["app", "data"]:
        for file in (PROJECT / folder).rglob("*"):
            if file.is_file() and "__pycache__" not in file.parts and file.suffix != ".pyc":
                archive.write(file, file.relative_to(PROJECT))
print("Download:", CHECKPOINT_OUT)

if client is not None:
    client.__exit__(None, None, None)
    client = None